# Module 06 — UDFs & Planner Visibility

In [1]:
import sys
sys.path.insert(0, '/workspace')
from shared.spark_session import begin_module, ui_links, explain_plan
from shared import config
from shared.utils import timed_action, timed_read, drop_iceberg_table, reset_spark_configs

spark = begin_module('module-06-udfs')
ui_links()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 10:58:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


{'master_ui': 'http://localhost:8080',
 'app_ui': 'http://localhost:4040',
 'worker_1_ui': 'http://localhost:8081',
 'worker_2_ui': 'http://localhost:8082',
 'app_ui_internal': 'http://jupyter-lab:4040'}

In [2]:
from pyspark.sql.functions import udf, col, when, length
from pyspark.sql.types import DoubleType
trips = spark.read.parquet(*config.yellow_taxi_paths())
df = trips.select('PULocationID', 'total_amount').filter('PULocationID IS NOT NULL').limit(500_000)

## Python UDF — opaque to optimizer

In [3]:
@udf(DoubleType())
def fare_tax_py(amount):
    return float(amount) * 1.08 if amount else 0.0

ud = df.withColumn('fare_taxed', fare_tax_py(col('total_amount')))
ud.filter('fare_taxed > 50').explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (10)
+- Filter (9)
   +- GlobalLimit (8)
      +- Exchange (7)
         +- LocalLimit (6)
            +- Project (5)
               +- BatchEvalPython (4)
                  +- LocalLimit (3)
                     +- Filter (2)
                        +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [PULocationID#7, total_amount#16]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
PushedFilters: [IsNotNull(PULocationID)]
ReadSchema: struct<PULocationID:int,total_amount:double>

(2) Filter
Input [2]: [PULocationID#7, total_amount#16]
Condition : isnotnull(PULocationID#7)

(3) LocalLimit
Input [2]: [PULocationID#7, total_amount#16]
Arguments: 500000

(4) BatchEvalPython
Input [2]: [PULocationID#7, total_amount#16]
Arguments: [fare_tax_py(total_amount#16)#20], [pythonUDF0#22]

(5) Project
Output [3]: [PULocationID#7, total_amount#16, pythonUDF0#22 AS fare_taxed#21]
Input [3]: [PULo

## Native expression — same logic, optimizable

In [5]:
native = df.withColumn('fare_taxed', col('total_amount') * 1.08)
native.filter('fare_taxed > 50').explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Filter (7)
   +- GlobalLimit (6)
      +- Exchange (5)
         +- LocalLimit (4)
            +- Project (3)
               +- Filter (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [PULocationID#7, total_amount#16]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
PushedFilters: [IsNotNull(PULocationID)]
ReadSchema: struct<PULocationID:int,total_amount:double>

(2) Filter
Input [2]: [PULocationID#7, total_amount#16]
Condition : isnotnull(PULocationID#7)

(3) Project
Output [3]: [PULocationID#7, total_amount#16, (total_amount#16 * 1.08) AS fare_taxed#29]
Input [2]: [PULocationID#7, total_amount#16]

(4) LocalLimit
Input [3]: [PULocationID#7, total_amount#16, fare_taxed#29]
Arguments: 500000

(5) Exchange
Input [3]: [PULocationID#7, total_amount#16, fare_taxed#29]
Arguments: SinglePartition, ENSURE_REQUIREMENTS, [plan_id=187]

(6) GlobalLimit
Input 

## Benchmark

In [6]:
_, t_udf = timed_action(ud.filter('fare_taxed > 50').select('fare_taxed'))
_, t_native = timed_action(native.filter('fare_taxed > 50').select('fare_taxed'))
print(f'Python UDF: {t_udf:.2f}s')
print(f'native:     {t_native:.2f}s')

Python UDF: 0.84s
native:     0.30s


## Pandas UDF (Arrow vectorized)

In [7]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf(DoubleType())
def fare_tax_pandas(s: pd.Series) -> pd.Series:
    return s * 1.08

pd_df = df.withColumn('fare_taxed', fare_tax_pandas(col('total_amount')))
_, t_pd = timed_action(pd_df.filter('fare_taxed > 50').select('fare_taxed'))
print(f'pandas UDF: {t_pd:.2f}s')

pandas UDF: 0.80s
